<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/13_scaling_up_and_reproducibility_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Scaling Up and Reproducibility

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

The notebooks in this course run in minutes on a few thousand documents. Real projects are bigger,
run for hours or days, and have to give the same results a year later, on someone else's computer,
when a reviewer asks. This notebook is about the habits that make that possible.

A true story to start with. While preparing this course in October 2026, three things broke in the
course's own notebooks without a single change to our code:

- **transformers 5** no longer accepted the `warmup_ratio` argument we used for fine-tuning BERT;
- a change in how the tokenizers library returns **overlapping chunks** of long texts silently gave
  us the wrong windows;
- **pandas 3** changed what `groupby(...).apply(...)` returns, which broke our data loading.

All three came from updated packages, because Colab updates its pre-installed versions regularly.
Code that isn't pinned to versions will break; the only question is when.

In this notebook you will:

- record your software environment and write a pinned `requirements.txt`
- pin models and data to exact versions, and fingerprint your input files
- control randomness, and report how much results vary with it
- organize a project so that it can be rerun from raw data to results
- cache expensive steps, process data in chunks, and make long jobs resumable
- turn a notebook into a script and run it on a computing cluster
- share code and data so that others can reproduce your work

**Contents**

1. Record your environment
2. Pin models and data
3. Randomness
4. Project structure
5. Caching expensive steps
6. Big data in chunks, and jobs that survive a crash
7. From notebook to script
8. Running on a cluster
9. Sharing: version control, archives and a checklist

No GPU needed.

There are 5 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
import hashlib
import json
import logging
import os
import platform
import random
import subprocess
import sys
import time
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd

WORK = Path("bvt_project")
WORK.mkdir(exist_ok=True)
print("Python", platform.python_version(), "on", platform.platform())

## 1. Record your environment

The minimum: the Python version and the exact version of every package you use. `importlib.metadata`
reads installed versions without importing the packages.

In [ ]:
PACKAGES = ["numpy", "pandas", "scikit-learn", "torch", "transformers", "sentence-transformers",
            "spacy", "matplotlib"]

def environment_report(packages):
    rows = []
    for p in packages:
        try:
            rows.append({"package": p, "version": metadata.version(p)})
        except metadata.PackageNotFoundError:
            rows.append({"package": p, "version": "not installed"})
    return pd.DataFrame(rows)

env = environment_report(PACKAGES)
env

A **`requirements.txt`** with pinned versions (`package==version`) lets anyone recreate the
environment with `pip install -r requirements.txt`. Pin what you *import*; pip resolves the rest.
(`pip freeze` lists *everything* installed, which on Colab is hundreds of packages, many of them
irrelevant and some impossible to install elsewhere.)

In [ ]:
pinned = [f"{r.package}=={r.version}" for r in env.itertuples() if r.version != "not installed"]
(WORK / "requirements.txt").write_text("\n".join(pinned) + "\n")
print((WORK / "requirements.txt").read_text())

Version differences are also something your code can check for. The course's BERT notebook needed one
setting that works in both transformers 4 and 5; a check like this makes the assumption explicit and
fails loudly instead of silently:

In [ ]:
from packaging.version import Version

def require(package, minimum=None, below=None):
    v = Version(metadata.version(package))
    if minimum and v < Version(minimum):
        raise RuntimeError(f"{package} {v} is older than {minimum}")
    if below and v >= Version(below):
        raise RuntimeError(f"{package} {v} is newer than tested (< {below})")
    print(f"{package} {v}: OK")

require("pandas", minimum="2.0")
require("numpy", minimum="1.24")

> **✏️ Exercise 1**
>
> The name you `import` isn't always the name you `pip install`: `sklearn` comes from
> `scikit-learn`, `bs4` from `beautifulsoup4`, `fitz` from `PyMuPDF`. Write a function
> `requirements_for(import_names)` that turns a list of import names into pinned requirement lines,
> using `metadata.packages_distributions()` (which maps import names to distribution names). Test it on
> `["numpy", "pandas", "sklearn", "matplotlib", "yaml", "PIL"]`.

In [ ]:
# Your code here

For more than a list of packages: **uv** or **conda** create isolated environments from a lock file
that pins every dependency, and **Docker** captures a whole operating system. For most papers, a pinned
`requirements.txt` plus the Python version is a big step up from nothing.

## 2. Pin models and data

Models and datasets on the Hugging Face Hub are git repositories: they can be updated under the same
name. `revision=` pins an exact commit:

```python
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",
                                                           revision="<commit hash>")
hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                revision="refs/convert/parquet", filename="default/train/0000.parquet")
```

(The course notebooks use `revision="refs/convert/parquet"`, a branch, which pins *which version of
the files* but can still move. A commit hash can't.) To find the current commit of a model:

In [ ]:
def current_commit(repo_id, repo_type="model"):
    try:
        from huggingface_hub import HfApi
        return HfApi().repo_info(repo_id, repo_type=repo_type).sha
    except Exception as e:                       # offline, or the hub is unreachable
        return f"unavailable ({type(e).__name__})"

print("distilbert-base-uncased:", current_commit("distilbert-base-uncased"))

For **LLM APIs** (Day 4), record the exact model version string and the date: providers update and
retire models, and the same name can behave differently months later. That's one more argument for open
models you can download and keep.

For **your own data files**, store a fingerprint: a SHA-256 checksum changes if even one byte changes,
so you can prove (and check) that results were computed from exactly this file.

In [ ]:
def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

# a small example data file
raw = WORK / "data" / "raw"
raw.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": range(5), "text": ["a", "b", "c", "d", "e"]}).to_csv(raw / "example.csv", index=False)

manifest = {p.name: {"sha256": sha256(p), "bytes": p.stat().st_size} for p in raw.glob("*")}
(WORK / "data" / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## 3. Randomness

Train/test splits, neural network initialization, dropout, UMAP, sampling a gold-standard set:
randomness is everywhere. Setting a **seed** makes a run repeatable. Each library has its own random
number generator, so a helper that seeds all of them is useful:

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(42); a = np.random.rand(3)
set_seed(42); b = np.random.rand(3)
set_seed(43); c = np.random.rand(3)
print(a, b, c, sep="\n")

Two caveats:

- **GPUs** aren't always deterministic: some operations give slightly different results from run to
  run even with the same seed. `torch.use_deterministic_algorithms(True)` forces determinism where
  possible, at a cost in speed.
- A seed makes a result **repeatable**, not **robust**. One seed is one draw. If a different seed
  changes your conclusion, that's important to know (Bonus 11's stability check). Report results over
  several seeds: mean and spread.

> **✏️ Exercise 2**
>
> How much does a result depend on chance? Using the toy data below (numeric features standing in for
> TF-IDF vectors; or use any classification task from the course), train the same logistic regression
> model with 10 different random train/test splits (`random_state` 0-9). Report the mean and standard deviation of
> the accuracy, and the range. Would you trust a difference of 1 percentage point between two models
> evaluated on a single split?
>
> ```python
> from sklearn.datasets import make_classification   # stand-in: numeric features
> X, y = make_classification(n_samples=1500, n_features=40, n_informative=8, random_state=0)
> ```

In [ ]:
# Your code here

## 4. Project structure

A layout that many research projects use, with one rule above all: **raw data is read-only**.
Everything else can be regenerated from it by code.

```
my_project/
├── README.md            what this is, how to run it, in which order
├── requirements.txt     pinned packages
├── config.json          settings: paths, model names, seeds, thresholds
├── data/
│   ├── raw/             original data, never edited (or a script that downloads it)
│   ├── processed/       produced by code
│   └── MANIFEST.json    checksums of the raw files
├── notebooks/           exploration
├── src/                 reusable code and scripts
└── results/             tables and figures, produced by code
```

Use **relative paths** (`data/raw/...`), never `C:/Users/denise/Desktop/...`, and keep settings in a
**config file** rather than scattered through the code: then a change of model or threshold is one
edit, and the config documents the run.

In [ ]:
for d in ["data/processed", "notebooks", "src", "results"]:
    (WORK / d).mkdir(parents=True, exist_ok=True)

config = {"seed": 42, "embedding_model": "all-MiniLM-L6-v2",
          "embedding_model_revision": "<commit hash>", "min_cluster_size": 20,
          "data": "data/raw/example.csv"}
(WORK / "config.json").write_text(json.dumps(config, indent=2))
(WORK / "README.md").write_text("# My project\n\n1. `pip install -r requirements.txt`\n"
                                "2. `python src/process.py --config config.json`\n")

for p in sorted(WORK.rglob("*")):
    print("    " * (len(p.relative_to(WORK).parts) - 1) + p.name + ("/" if p.is_dir() else ""))

## 5. Caching expensive steps

Embedding a corpus, calling an LLM, OCR-ing a scan collection: some steps take hours, and you don't
want to repeat them every time you rerun the analysis. **Cache** the result on disk. The cache must
know *which settings* produced the result, or you'll load stale results after changing something. A
simple, robust approach: put a hash of the settings in the file name.

In [ ]:
CACHE = WORK / "cache"
CACHE.mkdir(exist_ok=True)

def settings_hash(**settings):
    return hashlib.sha1(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:10]

def cached_array(name, compute, **settings):
    path = CACHE / f"{name}_{settings_hash(**settings)}.npy"
    if path.exists():
        print(f"loading {path.name}")
        return np.load(path)
    print(f"computing {path.name} ...")
    result = compute(**settings)
    np.save(path, result)
    return result


def slow_embedding(n_docs, dim, seed):            # stands in for an hour of sentence-transformer work
    time.sleep(2)
    return np.random.default_rng(seed).normal(size=(n_docs, dim))

for _ in range(2):
    start = time.time()
    E = cached_array("embeddings", slow_embedding, n_docs=1000, dim=384, seed=1)
    print(f"  {E.shape} in {time.time() - start:.2f} s")
E = cached_array("embeddings", slow_embedding, n_docs=1000, dim=384, seed=2)   # new settings: recompute

`joblib.Memory` does the same automatically for any function (it hashes the arguments), and is worth
knowing for larger projects.

> **✏️ Exercise 3**
>
> Write `cached_table(name, compute, **settings)`, the same idea for DataFrames, saved as Parquet.
> Test it with a "compute" function that takes a few seconds (e.g., makes a table of `n` random
> documents with their length). Check that (1) the second call is instant, (2) changing a setting
> recomputes, and (3) the cache folder holds one file per combination of settings.

In [ ]:
# Your code here

## 6. Big data in chunks, and jobs that survive a crash

When a corpus doesn't fit in memory, or a job takes many hours, process it **in chunks** and **write
each chunk's result to disk as soon as it's done**. Then:

- memory use stays constant, however big the corpus;
- a crash (out of memory, a lost connection, a Colab timeout, one corrupt file) costs one chunk, not
  everything: the job **resumes** by skipping chunks that are already done;
- chunks can run **in parallel** on a cluster (Section 8).

Colab sessions disconnect after a while, so this matters even for medium-sized jobs there. Save to
Google Drive so results survive the session.

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("bvt")

corpus = pd.DataFrame({"id": range(1000),
                       "text": [f"document {i} about {'nitrogen' if i % 3 else 'water'} policy" for i in range(1000)]})
corpus.to_parquet(WORK / "data" / "raw" / "corpus.parquet", index=False)


def process(texts):
    # The expensive step (an LLM call, embeddings, OCR...); here just a word count
    return [len(t.split()) for t in texts]


def run_in_chunks(df, out_dir, chunk_size=100, fail_at=None):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n_chunks = (len(df) + chunk_size - 1) // chunk_size
    for i in range(n_chunks):
        part = out_dir / f"part-{i:04d}.parquet"
        if part.exists():
            continue                                         # done in an earlier run
        if i == fail_at:
            raise RuntimeError(f"simulated crash in chunk {i}")
        chunk = df.iloc[i * chunk_size:(i + 1) * chunk_size]
        result = chunk[["id"]].assign(n_words=process(chunk["text"].tolist()))
        tmp = part.with_suffix(".tmp")
        result.to_parquet(tmp, index=False)
        tmp.rename(part)                                     # appears only when complete
        log.info("chunk %d/%d done", i + 1, n_chunks)


out = WORK / "data" / "processed" / "word_counts"
try:
    run_in_chunks(corpus, out, fail_at=4)
except RuntimeError as e:
    log.error(e)
print("parts on disk after the crash:", sorted(p.name for p in out.glob("*.parquet")))

Writing to a temporary file and renaming it at the end means a half-written file never looks like a
finished chunk. Now we rerun after "fixing" the problem: it picks up where it stopped.

In [ ]:
run_in_chunks(corpus, out)
result = pd.concat([pd.read_parquet(p) for p in sorted(out.glob("part-*.parquet"))], ignore_index=True)
print(len(result), "rows; ids complete:", result["id"].tolist() == list(range(1000)))

To *read* big data in chunks: `pd.read_csv(path, chunksize=10_000)` returns an iterator of
DataFrames; `pyarrow.parquet.ParquetFile(path).iter_batches()` does the same for Parquet; and the
Hugging Face `datasets` library can stream a dataset without downloading it all (`streaming=True`).
For data that's larger than memory but needs SQL-style queries, **DuckDB** runs queries directly on
Parquet and CSV files.

> **✏️ Exercise 4**
>
> Real failures are often caused by *one bad document*, not a crash of the whole job. Change
> `run_in_chunks` (call it `run_in_chunks_tolerant`) so that `process` is applied per document inside a
> `try`/`except`: a document that fails gets `n_words = None` and its id and error message are appended
> to an `errors.csv` in the output folder. Test it with a `process` function that raises an error for
> every document whose id is divisible by 250.

In [ ]:
# Your code here

## 7. From notebook to script

Notebooks are great for exploring, less so for long runs: they're hard to run unattended, and their
cells can be run out of order. For the heavy, final steps, a **script** with command-line arguments and
logging is more robust. `%%writefile` saves a cell as a file:

In [ ]:
%%writefile bvt_project/src/process_chunk.py
"""Process one chunk of a corpus: python process_chunk.py --input X --output DIR --chunk 3 --n-chunks 10"""
import argparse
import logging
from pathlib import Path

import pandas as pd


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", required=True, help="Parquet file with columns id, text")
    parser.add_argument("--output", required=True, help="folder for the result parts")
    parser.add_argument("--chunk", type=int, required=True, help="which chunk (0-based)")
    parser.add_argument("--n-chunks", type=int, required=True, help="total number of chunks")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    out = Path(args.output) / f"part-{args.chunk:04d}.parquet"
    if out.exists():
        logging.info("chunk %d already done, skipping", args.chunk)
        return

    df = pd.read_parquet(args.input)
    chunk = df.iloc[args.chunk::args.n_chunks]                    # every n-th row: balanced chunks
    logging.info("chunk %d/%d: %d documents", args.chunk, args.n_chunks, len(chunk))
    result = chunk[["id"]].assign(n_words=chunk["text"].str.split().str.len())

    out.parent.mkdir(parents=True, exist_ok=True)
    result.to_parquet(out.with_suffix(".tmp"), index=False)
    out.with_suffix(".tmp").rename(out)
    logging.info("wrote %s", out)


if __name__ == "__main__":
    main()

In [ ]:
for chunk in [0, 1]:
    r = subprocess.run([sys.executable, "bvt_project/src/process_chunk.py",
                        "--input", "bvt_project/data/raw/corpus.parquet",
                        "--output", "bvt_project/results/script_parts",
                        "--chunk", str(chunk), "--n-chunks", "4"], capture_output=True, text=True)
    print(r.stderr.strip())

(In a notebook you can also write `!python bvt_project/src/process_chunk.py --input ...`; we use
`subprocess` here so the notebook shows the log output neatly.)

## 8. Running on a cluster

When a laptop or Colab isn't enough (days of GPU time, millions of documents), use a **high-performance
computing (HPC) cluster**. Dutch researchers have access to **Snellius**, the national supercomputer run
by SURF (compute time through a SURF or NWO application), and many universities run their own
cluster; at WUR, ask your group or IT about access to the university's HPC cluster (Anunna).

Clusters use a **job scheduler**, usually **SLURM**: you don't run programs directly, you submit a job
script that says what resources you need, and the scheduler runs it when they're free. A **job array**
runs the same script many times with a different index, which is exactly what our chunked script
needs:

In [ ]:
slurm = '''#!/bin/bash
#SBATCH --job-name=bvt-chunks
#SBATCH --time=01:00:00            # wall time per task (hh:mm:ss)
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --array=0-9                # 10 tasks, SLURM_ARRAY_TASK_ID = 0..9
#SBATCH --output=logs/%x_%A_%a.out # one log file per task
# For a GPU, add e.g.:  #SBATCH --gpus=1  and the right --partition (names differ per cluster)

module load Python/3.11            # module names differ per cluster: see `module avail`
source $HOME/venvs/bvt/bin/activate

python src/process_chunk.py --input data/raw/corpus.parquet --output results/parts \\
    --chunk $SLURM_ARRAY_TASK_ID --n-chunks 10
'''
(WORK / "run_chunks.slurm").write_text(slurm)
print(slurm)

On the cluster: `sbatch run_chunks.slurm` submits it, `squeue -u $USER` shows your jobs, `scancel`
cancels. Because finished chunks are skipped, you can resubmit the same array after a failure and only
the missing chunks do real work. Partition names, module names, limits and GPU types differ between
clusters, so start from your cluster's documentation and its example scripts.

> **✏️ Exercise 5**
>
> Run `process_chunk.py` for **all** chunks of a 4-chunk run (the loop above did 0 and 1), then
> combine the parts and check that every document id appears exactly once. Then edit the SLURM script
> text for a job that processes the corpus in **25** chunks with at most **5 running at the same
> time** (look up the `%` syntax of `--array`) and 30 minutes per task.

In [ ]:
# Your code here

## 9. Sharing: version control, archives and a checklist

### Version control

**git** records every version of your code, with a message about what changed and why. You can go
back, compare, and see which version produced which result. **GitHub** (or GitLab) hosts the repository
and makes collaboration easy; the course repository is one. GitHub Desktop is the easiest start (see
the slide on downloading the repository). Two habits:

- commit small, logical steps with clear messages ("Fix leakage in the train/test split", not
  "update");
- keep data, models, caches and secrets **out** of git with a `.gitignore` file.

In [ ]:
gitignore = '''# data and generated files: regenerate them, or archive them elsewhere
data/
results/
cache/
*.npy
*.parquet
models/

# secrets: never commit API keys
.env
*.key

# Python and notebook clutter
__pycache__/
.ipynb_checkpoints/
'''
(WORK / ".gitignore").write_text(gitignore)
print(gitignore)

### Archiving

A GitHub repository can change or disappear. For a publication, archive a **snapshot**:

- **Zenodo** gives a permanent DOI to a release of a GitHub repository (there's a direct GitHub
  integration), and to datasets.
- **OSF** (Open Science Framework) hosts pre-registrations, materials and data, with DOIs.
- Your university's data repository (for WUR: ask the library about **4TU.ResearchData** and
  **DANS**) is often the right place for data with access restrictions.

Text data often **can't be shared** in full (copyright, privacy, platform terms). Share what you can:
document IDs or URLs, the code that collects and processes the data, derived data (counts, topic
proportions, embeddings with care), and a description of the rest. Bonus 3 and 4 have more on the legal
side.

### Reproducibility checklist

- [ ] `README.md`: what, how to run, in which order, how long it takes, which hardware
- [ ] pinned `requirements.txt` and the Python version
- [ ] models and datasets pinned to exact versions (commit hashes, API model strings and dates)
- [ ] checksums of raw data files; raw data never edited by hand
- [ ] seeds set; results reported over several seeds where randomness matters
- [ ] settings in a config file; no absolute paths
- [ ] expensive steps cached, long jobs chunked and resumable, failures logged
- [ ] code in git; a tagged release archived with a DOI
- [ ] data shared as far as legally possible, with documentation of what isn't
- [ ] someone else (or you, on a fresh Colab) has rerun it from scratch

The last item is the real test. Everything else makes it pass.

### Further reading

- Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L., & Teal, T. K. (2017). Good enough
  practices in scientific computing. *PLOS Computational Biology, 13*(6), e1005510.
- The Turing Way Community. *The Turing Way: A handbook for reproducible, ethical and collaborative
  data science* (free online handbook).